In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader

from PIL import Image

import cuml
from cuml.svm import SVR

/usr/local/lib/python3.11/dist-packages/cudf/utils/_ptxcompiler.py:64: UserWarning: Error getting driver and runtime versions:

stdout:



stderr:

Traceback (most recent call last):
  File "<string>", line 4, in <module>
  File "/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/cudadrv/driver.py", line 314, in __getattr__
    raise CudaSupportError("Error at driver init: \n%s:" %
numba.cuda.cudadrv.error.CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:


Not patching Numba
  warnings.warn(msg, UserWarning)


CUDARuntimeError: cudaErrorInsufficientDriver: CUDA driver version is insufficient for CUDA runtime version

In [2]:
directory = "/kaggle/input/petfinder-pawpularity-score"
train_df = pd.read_csv(os.path.join(directory, 'train.csv'))

test_df = pd.read_csv(os.path.join(directory, 'test.csv'))

print('Train samples: ', len(train_df),
      '\nTrain samples: ', len(test_df), '\n')


Train samples:  8920 
Train samples:  992 



In [3]:

# Clip - ViT
class dataset(Dataset):
    def __init__(self, df, directory, model, processor, device, test=False):
        self.df = df
        self.test = test
        self.processor = processor
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self):
        split = 'train' if not self.test else 'test'
        x = []
        if not self.test:
            y = []
        
        for index, row in tqdm(self.df.iterrows(), total=len(self.df)):
            filename = os.path.join(directory, split, row.Id+'.jpg')
            
            image = self.processor(images=Image.open(filename), return_tensors="pt", padding=True)
            with torch.no_grad():
                image_features = self.model.get_image_features(**image.to(device)).squeeze()
            x.append(image_features.cpu().detach().numpy())
            
            if not self.test: 
                score = torch.FloatTensor([row.Pawpularity/100])
                y.append(score.cpu().detach().numpy())
                
            
        return (np.array(x), np.array(y).squeeze()) if not self.test else np.array(x)


In [4]:
train_df

,Id,Subject Focus,Eyes,Face,Near,Action,Accessory,Group,Collage,Human,Occlusion,Info,Blur,Pawpularity
0,1a8795e64a294ed0c95132e18ee198e1,0,1,1,1,0,0,0,0,0,0,0,0,55
1,08ce774252e822838bf598aa10518a11,0,1,1,1,0,0,0,0,0,0,0,0,100
2,a71c677d87606a56160b70c5f6fa4526,0,1,1,0,0,0,0,0,0,0,0,0,27
3,11d5914ad80403375c8e0ccd959fc08e,0,1,1,1,0,0,0,0,0,0,0,0,51
4,c04034bbfe0a0d89729541c2271a674e,0,1,1,1,0,0,0,0,0,0,0,0,48
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8915,ee97d86d26506239666d3679ff4a73aa,0,0,1,1,0,0,0,0,0,0,0,1,31
8916,7cf13d655a8d0d6a1d636c3920d49e86,0,0,1,1,0,0,0,0,0,0,0,0,33
8917,547e443edbf94b3646ff110324846bae,0,1,1,0,0,0,1,0,0,0,0,0,27
8918,fe2b587f649b9780900246c4b174a231,0,1,1,1,0,0,1,0,0,0,0,0,27


In [5]:

# --- Uncomment: for Clip to use ---
from transformers import CLIPModel, CLIPProcessor

# Adjust the paths to point to your extracted files
model_path = "/kaggle/input/clip-vit/pytorch/b-32-laion2b-s34b-b79k/1"
#model_path = "/kaggle/input/clip-vit/pytorch/l-14-datacomp-xl-s13b-b90k/1"

model_clip = CLIPModel.from_pretrained(model_path)
processor_clip = CLIPProcessor.from_pretrained(model_path)


2026-04-28 08:51:02.524496: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777366262.533868      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777366262.537009      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 08:51:02.548996: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/input/clip-vit/pytorch/b-32-laion2b-s34b-b79k/1'. Use `repo_type` argument if needed.

In [6]:

class dataset():
    def __init__(self, df, directory, processor, test=False):
        self.df = df
        self.directory = directory
        self.processor = processor
        self.test = test
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        split = 'test' if self.test else 'train'
        filename = self.df.Id[idx]
        address = os.path.join(self.directory, split, filename+'.jpg')
        image = self.processor(images=Image.open(address), return_tensors="pt", padding=True)
        for key, val in image.items():
            image[key] = val.squeeze()
            
        return image
        

In [7]:
directory = '/kaggle/input/petfinder-pawpularity-score'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_dataset = dataset(train_df, directory, processor_clip, test=False)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=False)

test_dataset = dataset(test_df, directory, processor_clip, test=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

NameError: name 'processor_clip' is not defined

In [8]:
X = []
model_clip = model_clip.to(device)
for img in tqdm(train_dataloader):
    x = model_clip.get_image_features(**img.to(device))
    #print('img: ', img.pixel_values.shape, 'Clip out: ', x.shape)
    
    X.append(x.cpu().detach().numpy())

X = np.concatenate(X, axis=0)
print('final X:', X.shape)

y = train_df.Pawpularity.values

NameError: name 'model_clip' is not defined

In [9]:
X_test = []
for img in tqdm(test_dataloader):
    x = model_clip.get_image_features(**img.to(device))
    #print('img: ', img.pixel_values.shape, 'Clip out: ', x.shape)
    
    X_test.append(x.cpu().detach().numpy())

X_test = np.concatenate(X_test, axis=0)
print('final X_test:', X_test.shape)

NameError: name 'test_dataloader' is not defined

In [10]:
#y = y.clip(1, 85)

In [11]:
print(X.shape)
print(X_test.shape)

AttributeError: 'list' object has no attribute 'shape'

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(np.vstack((X, X_test)))
X = scaler.transform(X)
print(X.shape)

ValueError: Found array with 0 feature(s) (shape=(2, 0)) while a minimum of 1 is required by StandardScaler.

In [13]:
reg = SVR(C=16.0, kernel='rbf', degree=3, max_iter=400000, output_type='numpy')
reg.fit(X, y)

NameError: name 'SVR' is not defined

In [14]:

print("Predicted values:", reg.predict(X[:10, :])) 
print("True values: ", y[:10])


NameError: name 'reg' is not defined

In [15]:

X_test = scaler.transform(X_test)
print('X shape: ', X_test.shape)


NotFittedError: This StandardScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [16]:
y_pred = reg.predict(X_test)
print("Predicted values:", y_pred) 


NameError: name 'reg' is not defined

In [17]:
# Prepare the submission DataFrame
submit = pd.DataFrame()
submit['Id'] = test_df['Id']
submit['Pawpularity'] = y_pred

# Save the submission to a CSV file
submit.to_csv('submission.csv', index=False)

NameError: name 'y_pred' is not defined

In [18]:
submit

,Id
0,ee51b99832f1ba868f646df93d2b6b81
1,caddfb3f8bff9c4b95dbe022018eea21
2,582eeabd4a448a53ebb79995888a4b0b
3,afc1ad7f0c5eea880759d09e77f7deee
4,d5bdf3446e86ce4ec67ce7a00f1cccc2
...,...
987,46bcb1c4ca0b78c584d7a65e1b4b2590
988,8064261ce5f7197fdc849641b3192eef
989,a5ff4923e393fa08e67da31c7eb4edd5
990,fa6a53f962225c17b22392748f91f9ac
